# Krussel Smith Implementation

Here is where I give a whack at getting Krussel Smith going. I am trying not to crash out (interesting strategy, let's see how it plays out).

In [1]:
using Pkg
for pkg in ["JLD2", "StatsBase", "Printf", "LinearAlgebra", "Dates"]
    if Base.find_package(pkg) === nothing
        Pkg.add(pkg)
    end
end

In [2]:
using JLD2
using Random
using Printf
using LinearAlgebra: dot
using StatsBase: countmap

include("src/ModelTypes.jl")
@printf("Loaded ModelTypes.jl\n")

include("src/Compute.jl")
@printf("Loaded Compute.jl\n")

include("src/ModelFunctions.jl")
@printf("Loaded ModelFunctions.jl\n")

include("src/EGM.jl")
@printf("Loaded EGM.jl\n")

include("src/DistrTools.jl")
@printf("Loaded DistrTools.jl\n")

include("src/Solvers.jl")
@printf("Loaded Solvers.jl\n")

include("src/SteadyState.jl")
@printf("Loaded Steady States.jl\n")

include("src/Predict.jl")
@printf("Loaded Predict.jl\n")

using .ModelTypes
using .ModelFunctions
using .Compute
using .EGM
using .DistrTools
using .Solvers
using .SteadyState
using .Predict

Loaded ModelTypes.jl
Loaded Compute.jl
Loaded ModelFunctions.jl
Loaded EGM.jl
Loaded DistrTools.jl
Loaded Solvers.jl
Loaded Steady States.jl
Loaded Predict.jl


In [3]:
# model parameters
const α::Float64 = 0.36;
const β::Float64 = 0.96;
const δ::Float64 = 0.06;
const σ::Float64 = 2;
const ϕ::Float64 = 0;

# grid sizes and parameters
const na::Int64 = 100; 
const nl::Int64 = 7;
const nz::Int64 = 5;
const nk::Int64 = 25;

const a_l::Float64 = 0;
const a_h::Float64 = 100;

# calibrations for idiosyncratic income (vibes)
const μ_l::Float64 = 0; 
const ρ_l::Float64 = .9;
const σ_l::Float64 = .2;

# calibrations for aggregate TFP (Khan and Thomas 2013 would have 
# ρ_z = 0.909; I am setting it lower for two dimensional z with
# some variance for now.)
const μ_z::Float64 = 0;
const ρ_z::Float64 = .909;
const σ_z::Float64 = 0.014;


In [4]:

grid_range = 2.575;

π_l, lgrid = getTauchen(nl,  μ_l, σ_l, ρ_l, grid_range);
π_z, zgrid= getTauchen(nz,  μ_z, σ_z, ρ_z, grid_range);

stationary_l = stationary(π_l);
const lagg::Float64 = dot(stationary_l, lgrid);
 
agrid = logspace(a_l, a_h, na);
amu = collect(range(a_l, a_h, length=na*10));

const kL::Float64 = 6; const kH::Float64 = 12; # this could be informed by steady states, but for now we do it this way
Kgrid = collect(range(kL, kH, length = nk));

In [5]:
const np::Int64 = 10; # number of policies
const pol_l::Float64 = 0;
const pol_h::Float64 = 1;

τ_grid = range(pol_l, pol_h, length = np);
η_grid = range(pol_l, pol_h, length = np);

captax = repeat([0], outer = nl);

η = 0; τ = 0;

Okay, so the goal here is that there are forecasts across moments of the aggregate capital distribution, which means that I have to send new prices to the household across capital moments and solve each household problem at each aggregate capital moment; then I simulate, check the accuracy of the forecasts with the actual responses to a TFP shock path, and then update my forecasts. 

What this means is that every aggregate capital moment generates a price, which leads to the solution of the household problem for each aggregate capital moment with uncertainty over TFP. To do that, I need to make a couple changes: 

0. Choosing which steady state to start from--I need to import that stationary distribution. 
1. Set up a forecasting rule. Here, I'll just set it to something like $\log(K') = 0.05 a + .95\log(K)$ for each transition z -> z' (so this is nz^2 forecasting rules)
2. Simulating a shock path: I want to keep this constant, so it's setting the RNG seed so that it spits out the same z path each time.  
3. Then running regressions and updating--because I have effectively 4 rules, this should have several periods. 

In [6]:
const NT = 5000; #three thousand periods for sampling
const rnseed = 1234567;

zt = simz(NT, nz, rnseed, π_z);

# forecasting rules: 
# pulled from a previous sollution
const Kfore_start = [0.102898 0.946118; 
            0.112504 .944115;
            0.121485 0.942448;
            0.131148 0.940548;
            0.139579 0.939373];

5×2 Matrix{Float64}:
 0.102898  0.946118
 0.112504  0.944115
 0.121485  0.942448
 0.131148  0.940548
 0.139579  0.939373

Now to do the actual solve:

In [7]:
V = nothing
EV = nothing
G = nothing  
C = nothing
CI = nothing
LI = nothing
μ = nothing

foredist = 10;
const dTol = 1e-3;
const vTol = 1e-6;


r_vals = zeros(nk, nz); w_vals = zeros(nk, nz); λ_vals = zeros(nk, nz); 

for ik = 1:nk,  iz = 1:nz
    r_vals[ik, iz] = calcr(α, δ, Kgrid[ik], η, zgrid[iz])
    w_vals[ik, iz] = calcw(α, Kgrid[ik], η, zgrid[iz])
    denom = dot((w_vals[ik, iz] .* lgrid).^(1 - τ), stationary(π_l))
    tot_inc = w_vals[ik, iz] * dot(lgrid, stationary(π_l))
    λ_vals[ik, iz] = tot_inc / denom
end

const params = ModelParams(α, β, δ, σ, ϕ, agrid, 
    lgrid, zgrid, π_l, π_z, amu, Kgrid);

const policies = ProposedPolicies(η, τ, captax);

const prices = ImpliedRegimeParams_KS(λ_vals, r_vals, w_vals)

# init V:

V0 = zeros(nk,nz,nl,na); V  = zeros(nk,nz,nl,na);
EV = zeros(nk,nz,nl,na); G  = zeros(nk,nz,nl,na); 
G0 = zeros(nk, nz, nl, na); C  = zeros(nk,nz,nl,na);

for ik = 1:nk,  iz = 1:nz, il = 1:nl, ia = 1:na
    kval = agrid[ia];
    yval = (1 + r_vals[ik, iz]*(1-captax[il]))*kval + w_vals[ik, iz]*lgrid[il] - r_vals[ik, iz]*ϕ;
    ymin = max(1e-10, yval);
    V0[ik, iz, il, ia] = log(ymin);
    G0[ik, iz, il, ia] = agrid[ia];
end


In [8]:
Kfore, Kt = run_KS(V, V0, G, G0, C, params, policies, prices,
                zt, Kfore_start, vTol, dTol)

println(Kfore)

[0.10288897936077203 0.9461242240026471; 0.1124911315211353 0.9441220014049895; 0.12146794029335524 0.9424558318471554; 0.13112726172334643 0.9405566329661256; 0.13955835766697267 0.9393805299575719]


In [9]:
results = Dict{Tuple{Float64,Float64}, NamedTuple}()

# storing each stationary equilibrium
results[(η, τ)] = (
    Kfore,
    V = V,
    G = G,
    C = C,
    Kt = Kt,
    zt = zt,
    policies = policies
);

In [10]:
@save "../d/KS_solves.jld2" results params zt

In [11]:
print(π_z)

[0.8370276127980094 0.16294901698975173 2.3370211836626353e-5 4.022338018216942e-13 0.0; 0.03395354720350539 0.8628268622111023 0.10321285959633375 6.730989008829624e-6 4.973799150320701e-14; 1.7971484058070882e-6 0.061229388312376475 0.8775376290784354 0.061229388312376565 1.7971484057577314e-6; 4.977489407637338e-14 6.7309890088411185e-6 0.10321285959633363 0.8628268622111024 0.03395354720350541; 5.951816699063212e-25 4.02223261070338e-13 2.337021183662143e-5 0.1629490169897517 0.8370276127980094]